# 04 Model Training

This notebook trains baseline fraud detection models and compares their performance using fraud-focused metrics. Because the dataset is highly imbalanced, accuracy is not used as the main metric. The focus is on precision, recall, F1-score, ROC-AUC and PR-AUC.

In [1]:
from pathlib import Path
import joblib
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from xgboost import XGBClassifier

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = BASE_DIR / "data" / "processed"
MODEL_DIR = BASE_DIR / "models"
REPORTS_DIR = BASE_DIR / "reports"

MODEL_DIR.mkdir(exist_ok=True)
REPORTS_DIR.mkdir(exist_ok=True)

X_train = pd.read_csv(DATA_DIR / "X_train.csv")
X_test = pd.read_csv(DATA_DIR / "X_test.csv")
y_train = pd.read_csv(DATA_DIR / "y_train.csv").squeeze()
y_test = pd.read_csv(DATA_DIR / "y_test.csv").squeeze()

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)
print("Fraud distribution in training set:")
print(y_train.value_counts(normalize=True) * 100)

Training shape: (227845, 31)
Testing shape: (56962, 31)
Fraud distribution in training set:
Class
0    99.827075
1     0.172925
Name: proportion, dtype: float64


In [2]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1,
    ),
    "XGBoost": XGBClassifier(
        n_estimators=100,
        max_depth=5,
        learning_rate=0.1,
        random_state=42,
        eval_metric="logloss",
    ),
}

results = []
best_model = None
best_model_name = None
best_f1 = 0

for name, model in models.items():
    print(f"\n========== {name} ==========")
    model.fit(X_train, y_train)

    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1]

    precision = precision_score(y_test, predictions)
    recall = recall_score(y_test, predictions)
    f1 = f1_score(y_test, predictions)
    roc_auc = roc_auc_score(y_test, probabilities)
    pr_auc = average_precision_score(y_test, probabilities)
    cm = confusion_matrix(y_test, predictions)

    print(classification_report(y_test, predictions))
    print("Confusion Matrix:")
    print(cm)

    results.append({
        "model": name,
        "fraud_precision": precision,
        "fraud_recall": recall,
        "fraud_f1_score": f1,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
        "false_positives": int(cm[0][1]),
        "false_negatives": int(cm[1][0]),
        "true_frauds_detected": int(cm[1][1]),
    })

    if f1 > best_f1:
        best_f1 = f1
        best_model = model
        best_model_name = name

results_df = pd.DataFrame(results).sort_values(by="fraud_f1_score", ascending=False)
results_df


========== Logistic Regression ==========
              precision    recall  f1-score   support

           0       1.00      0.97      0.99     56864
           1       0.06      0.91      0.10        98

    accuracy                           0.97     56962
   macro avg       0.53      0.94      0.55     56962
weighted avg       1.00      0.97      0.98     56962

Confusion Matrix:
[[55350  1514]
 [    9    89]]

========== Random Forest ==========
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.96      0.72      0.83        98

    accuracy                           1.00     56962
   macro avg       0.98      0.86      0.91     56962
weighted avg       1.00      1.00      1.00     56962

Confusion Matrix:
[[56861     3]
 [   27    71]]

========== XGBoost ==========
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.92     

,model,fraud_precision,fraud_recall,fraud_f1_score,roc_auc,pr_auc,false_positives,false_negatives,true_frauds_detected
2,XGBoost,0.915663,0.775510,0.839779,0.948537,0.834953,7,22,76
1,Random Forest,0.959459,0.724490,0.825581,0.952955,0.861590,3,27,71
0,Logistic Regression,0.055521,0.908163,0.104644,0.973733,0.722214,1514,9,89


In [3]:
baseline_model_path = MODEL_DIR / "fraud_detection_model.pkl"
results_path = REPORTS_DIR / "baseline_model_results.csv"

joblib.dump(best_model, baseline_model_path)
results_df.to_csv(results_path, index=False)

print(f"Best baseline model: {best_model_name}")
print(f"Best fraud F1-score: {best_f1:.4f}")
print(f"Model saved to: {baseline_model_path}")
print(f"Results saved to: {results_path}")

Best baseline model: XGBoost
Best fraud F1-score: 0.8398
Model saved to: /mnt/c/Users/mbaik/Desktop/Alaid/Projects/credit-card-fraud-detection/models/fraud_detection_model.pkl
Results saved to: /mnt/c/Users/mbaik/Desktop/Alaid/Projects/credit-card-fraud-detection/reports/baseline_model_results.csv


## Interpretation

The best baseline model should be selected based on fraud-focused metrics, especially recall and F1-score. In fraud detection, false negatives are costly because they represent fraudulent transactions that the model failed to detect. The next stage improves the best model using class imbalance weighting and threshold tuning.